# 12 — Does an extra year of training history help? (optional experiment)

**Question:** does 2024–2025 training improve validation performance over the frozen 2025-only model?

After a successful 2024 audit in 05b, select your project kernel and **Run All**. No source-window, path or mode edits are needed. Earlier notebooks, active-run selection, the original model and dashboard inputs stay unchanged. No prepared test data or latest predictions are accessed.

The first execution downloads the 12 audited 2024 monthly files and caches only the selected basket plus provenance. Cached sources are verified and reused thereafter. Integration stops on changed source identities, unexpected egg units or duplicate source keys. Records missing required series metadata are retained in the source cache, audited and excluded only from supplementary training; no geography or shop type is guessed. No partial-year dataset is used. No new packages are needed.

**Fixed:** selected model settings, feature list, 10% target, original score cutoff and original validation features/labels/rows. The original 2025 item eligibility and retail policy remain fixed, deliberately isolating history within that basket; this does not optimise eligibility for the longer period. Original 2025 conservative exclusions remain as prepared. Invalid 2024 prices are removed by the same finite-positive rule; other suspicious values remain. No 09 exclusions or 10 features are added.

**Rebuilt:** training aggregates, exact-week history, labels and eligibility from 2024–2025. Training-only preprocessing is fitted again. Current lookup units/descriptions are a provisional interpretation of historical records, not independently verified historical metadata. That assumption must be reported.

This is post-hoc exploratory development on an already inspected validation period, not new independent evidence. Do not tune additional settings to these results or automatically replace the dashboard model.

**Historical checkpoint dependency:** references to the old combined Notebook 06 mean its saved forecasting checkpoint, not the new Logistic Regression notebook. The old source is in Git history. This optional extension requires its original saved artifacts and is not part of model notebooks 06–10.


**Active replacement for new training:** use root `09b_longer_history_lightgbm.ipynb` before 10. This older notebook remains provenance for the already published dashboard checkpoint.


In [3]:
# Open this notebook from the FYP folder or its research subfolder.
import sys
from pathlib import Path
REPOSITORY_ROOT = Path.cwd()
if not (REPOSITORY_ROOT / "pricecatcher_config.py").is_file():
    REPOSITORY_ROOT = REPOSITORY_ROOT.parent
if not (REPOSITORY_ROOT / "pricecatcher_config.py").is_file():
    raise FileNotFoundError("Open the FYP project folder before running this research notebook.")
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

import hashlib
import io
import json
from pathlib import Path
from urllib.request import Request, urlopen
from importlib import reload, metadata
import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score, confusion_matrix
import pricecatcher_config as cfg
cfg = reload(cfg)
if cfg.EXPERIMENT != "holdout":
    raise ValueError("This comparison is for the completed holdout-training experiment only.")

def file_hash(path):
    with Path(path).open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

model_dir = next((p for p in cfg.MODEL_DIRS if (p / "best_model.joblib").is_file()), None)
if model_dir is None:
    raise FileNotFoundError("Complete Notebook 06 first; its original model is the reference.")
manifest = json.loads((model_dir / "model_manifest.json").read_text())
selection = json.loads((model_dir / "selection.json").read_text())
if manifest.get("run_status") != "completed" or selection != manifest["selection"]:
    raise ValueError("A matching completed main experiment is required.")
if file_hash(model_dir / "best_model.joblib") != manifest["model_sha256"]:
    raise ValueError("Original model changed. Restore its original file.")
packages = ["pandas", "numpy", "scikit-learn", "joblib"]
if selection["family"] == "LightGBM":
    packages.append("lightgbm")
if selection["family"] == "XGBoost":
    packages += [p for p in ["xgboost", "xgboost-cpu"] if p in manifest["package_versions"]]
for package in packages:
    if metadata.version(package) != manifest["package_versions"][package]:
        raise ValueError(f"Use the saved model's {package} version before comparing histories.")
folder = cfg.PROJECT_ROOT / "experiments" / "holdout" / "05_modelling"
for name in ["experiment_settings.json", "train.parquet", "validation.parquet"]:
    if file_hash(folder / name) != manifest["input_sha256"][f"holdout/{name}"]:
        raise ValueError(f"Frozen {name} changed. Restore its original input.")
settings = json.loads((folder / "experiment_settings.json").read_text())
cleaning_path = cfg.STAGE3_DIR / "cleaning_summary.json"
cleaning = json.loads(cleaning_path.read_text())
current_source = cfg.verify_source_snapshot()
if cleaning["preparation"] != settings["preparation"] or settings["preparation"] != {
        **cfg.PREPARATION_SETTINGS, "source_manifest_sha256": file_hash(cfg.STAGE1_DIR / "source_manifest.json"),
        "source_snapshot_sha256": current_source["source_snapshot_sha256"]}:
    raise ValueError("Matching current cleaning, settings and source snapshot are required.")
training_end = pd.Timestamp(settings["train_end"])
validation_end = pd.Timestamp(settings["validation_end"])
features = settings["feature_columns"]
keys = cfg.GROUP_KEYS + ["week_start"]
audit_path = cfg.DATA_HOME / "11_historical_audit" / "monthly_availability.csv"
if not audit_path.exists():
    raise FileNotFoundError("Run Notebook 11 successfully on the Mac first.")
audit = pd.read_csv(audit_path)
months = pd.period_range("2024-01", "2024-12", freq="M").astype(str).tolist()
if (sorted(audit.month.tolist()) != months or not audit.availability.eq("available").all()
        or not audit.schema_compatible.eq(True).all() or audit.etag.isna().any()):
    raise ValueError("The audit must identify all 12 accessible, schema-compatible 2024 files and their ETags.")
for _, row in audit.iterrows():
    if row.url != cfg.BASE_PRICE_URL.format(month=row.month) or not 12 <= row.file_bytes <= 64 * 1024 * 1024:
        raise ValueError("Audit URL or file size is outside the declared integration contract.")
source_policy = {"schema_version": 2, "months": months,
    "missing_metadata_policy": "retain source; audit and quarantine missing required local-series fields from supplementary training",
    "monthly_sources": audit.sort_values("month")[["month", "url", "etag", "file_bytes"]].to_dict(orient="records"),
    "item_map": {str(k): v for k, v in cfg.PROJECT_ITEM_MAP.items()}, "egg_30_codes": cfg.EGG_30_CODES,
    "lookup_sha256": {name: file_hash(cfg.LOOKUP_DIR / name) for name in ["lookup_item.parquet", "lookup_premise.parquet"]},
    "historical_metadata": "provisional current frozen lookup interpretation; historical units not independently verified"}
source_id = hashlib.sha256(json.dumps(source_policy, sort_keys=True).encode()).hexdigest()[:16]
source_dir = cfg.DATA_HOME / "12_longer_history" / "2024_source" / source_id
source_dir.mkdir(parents=True, exist_ok=True)
source_path = source_dir / "selected_prices_2024.parquet"
source_manifest_path = source_dir / "source_manifest.json"
metadata_audit_path = source_dir / "historical_metadata_review.csv"
print("Original run:", cfg.PROJECT_ROOT)
print("2024 cache:", source_dir)
print("Fixed validation cutoff:", selection["score_cutoff"], "| Selected settings:", selection["candidate"])

Original run: /Users/alfredyap/Documents/GitHub/FYP/data/runs/run_20261019_013241_618240
2024 cache: /Users/alfredyap/Documents/GitHub/FYP/data/12_longer_history/2024_source/98056c71efcb631c
Fixed validation cutoff: 0.6006503045679725 | Selected settings: lgb_leaves31


## Integrate the audited year once

The monthly ETags and file sizes from 11 identify the audited source versions. HTTP conditional requests and SHA-256 record the files actually used. The provisional lookup interpretation is recorded in the manifest. Cached files avoid further downloads. Original 2025 data and notebook configuration are never rewritten.

In [4]:
if source_manifest_path.exists():
    source_record = json.loads(source_manifest_path.read_text())
    if source_record["policy"] != source_policy or not source_path.exists() or file_hash(source_path) != source_record["selected_sha256"] or not metadata_audit_path.exists() or file_hash(metadata_audit_path) != source_record["metadata_audit_sha256"]:
        raise ValueError("2024 cache changed. Restore its matching source snapshot; do not bypass hashes.")
    print("Reusing verified selected 2024 source:", source_record["selected_rows"], "rows")
else:
    item = pd.read_parquet(cfg.LOOKUP_DIR / "lookup_item.parquet")
    premise = pd.read_parquet(cfg.LOOKUP_DIR / "lookup_premise.parquet")
    for table, key in [(item, "item_code"), (premise, "premise_code")]:
        if table[key].isna().any() or table[key].duplicated().any():
            raise ValueError("Lookup codes must be unique and non-missing.")
    units = item.set_index("item_code").unit.astype("string").str.strip().str.lower()
    for item_code in cfg.ALL_EGG_CODES:
        if units.get(item_code) != ("30 biji" if item_code in cfg.EGG_30_CODES else "10 biji"):
            raise ValueError("Egg units disagree with the conversion contract.")
    item = item[["item_code", "item", "unit", "item_group", "item_category"]]
    premise = premise[["premise_code", "premise", "state", "district", "premise_type"]]
    frames, downloaded, metadata_reviews = [], [], []
    for row in source_policy["monthly_sources"]:
        print("Downloading audited month:", row["month"])
        request = Request(row["url"], headers={"If-Match": row["etag"], "Accept-Encoding": "identity"})
        with urlopen(request, timeout=60) as response:
            if response.status != 200 or response.headers.get("ETag") != row["etag"]:
                raise ValueError("Source identity changed. Preserve this experiment and repeat the 2024 audit.")
            payload = response.read(int(row["file_bytes"]) + 1)
        if len(payload) != int(row["file_bytes"]):
            raise ValueError("Monthly payload differs from the audited file size; no partial-year cache is committed.")
        prices = pd.read_parquet(io.BytesIO(payload), columns=["date", "premise_code", "item_code", "price"])
        prices["item_code"] = pd.to_numeric(prices.item_code, errors="raise").astype("Int64")
        prices = prices.loc[prices.item_code.isin(cfg.SCOPE_ITEM_CODES)].copy()
        prices["premise_code"] = pd.to_numeric(prices.premise_code, errors="raise").astype("Int64")
        prices["date"] = pd.to_datetime(prices.date, errors="raise").dt.normalize()
        prices["price"] = pd.to_numeric(prices.price, errors="coerce")
        if prices.empty or prices.date.isna().any() or not prices.date.dt.to_period("M").eq(pd.Period(row["month"])).all():
            raise ValueError("Selected records must be nonempty and belong to their audited month.")
        if prices.duplicated(["date", "premise_code", "item_code"]).any():
            raise ValueError("Duplicate monthly source keys need review; no automatic deduplication.")
        prices["project_group"] = prices.item_code.map(cfg.PROJECT_ITEM_MAP)
        data = prices.merge(item, on="item_code", how="left", validate="many_to_one")
        data = data.merge(premise, on="premise_code", how="left", validate="many_to_one")
        for column in cfg.TEXT_COLS:
            data[column] = data[column].astype("string").str.strip().replace("", pd.NA)
        # Keep source records intact; only model eligibility needs complete series identifiers.
        missing = data[cfg.GROUP_KEYS].isna()
        needs_review = missing.any(axis=1)
        if needs_review.any():
            review = data.loc[needs_review, ["date", "premise_code", "item_code"]].copy()
            review["missing_fields"] = missing.loc[needs_review].apply(
                lambda fields: ", ".join(fields.index[fields]), axis=1)
            review["lookup_status"] = np.where(review.premise_code.isin(premise.premise_code),
                "known shop code; required metadata incomplete", "shop code absent from frozen lookup")
            review["month"] = row["month"]
            metadata_reviews.append(review.groupby(["month", "premise_code", "item_code", "missing_fields", "lookup_status"],
                dropna=False).agg(source_records=("date", "size"), first_date=("date", "min"), last_date=("date", "max")).reset_index())
            print("Source rows needing metadata review:", int(needs_review.sum()), "(kept in cache; not eligible for training)")
        data.loc[data.item_code.isin(cfg.EGG_30_CODES), "price"] /= 3
        data.loc[data.item_code.isin(cfg.ALL_EGG_CODES), "unit"] = "10 biji"
        frames.append(data[cfg.FINAL_COLS])
        downloaded.append({"month": row["month"], "raw_sha256": hashlib.sha256(payload).hexdigest(),
            "etag": row["etag"], "selected_rows": len(data)})
        del prices, data, payload
    combined = pd.concat(frames, ignore_index=True)
    temporary = source_path.with_suffix(".tmp")
    combined.to_parquet(temporary, index=False)
    temporary.replace(source_path)
    metadata_review = (pd.concat(metadata_reviews, ignore_index=True) if metadata_reviews else
        pd.DataFrame(columns=["month", "premise_code", "item_code", "missing_fields", "lookup_status", "source_records", "first_date", "last_date"]))
    temporary = metadata_audit_path.with_suffix(".tmp")
    metadata_review.to_csv(temporary, index=False)
    temporary.replace(metadata_audit_path)
    source_record = {"policy": source_policy, "monthly_downloads": downloaded,
        "selected_rows": len(combined), "selected_sha256": file_hash(source_path),
        "metadata_audit_sha256": file_hash(metadata_audit_path),
        "source_rows_missing_required_metadata": int(metadata_review.source_records.sum())}
    temporary = source_manifest_path.with_suffix(".tmp")
    temporary.write_text(json.dumps(source_record, indent=2))
    temporary.replace(source_manifest_path)
    del frames, combined
    print("Committed the complete selected year; raw monthly files were not saved.")
print("Source rows missing required metadata:", source_record["source_rows_missing_required_metadata"])
print("Metadata evidence:", metadata_audit_path)
metadata_review = pd.read_csv(metadata_audit_path)
if not metadata_review.empty:
    display(metadata_review.head(20))


Source rows needing metadata review: 23 (kept in cache; not eligible for training)
Committed the complete selected year; raw monthly files were not saved.
Source rows missing required metadata: 23
Metadata evidence: /Users/alfredyap/Documents/GitHub/FYP/data/12_longer_history/2024_source/98056c71efcb631c/historical_metadata_review.csv


,month,premise_code,item_code,missing_fields,lookup_status,source_records,first_date,last_date
0,2024-02,20927,1,"state, district, premise_type",shop code absent from frozen lookup,1,2024-02-13,2024-02-13
1,2024-02,20927,92,"state, district, premise_type",shop code absent from frozen lookup,3,2024-02-05,2024-02-19
2,2024-02,20927,94,"state, district, premise_type",shop code absent from frozen lookup,3,2024-02-05,2024-02-19
3,2024-02,20927,113,"state, district, premise_type",shop code absent from frozen lookup,3,2024-02-05,2024-02-19
4,2024-02,20927,114,"state, district, premise_type",shop code absent from frozen lookup,3,2024-02-05,2024-02-19
5,2024-02,20927,917,"state, district, premise_type",shop code absent from frozen lookup,2,2024-02-13,2024-02-19
6,2024-02,20927,1110,"state, district, premise_type",shop code absent from frozen lookup,1,2024-02-19,2024-02-19
7,2024-02,20927,1111,"state, district, premise_type",shop code absent from frozen lookup,1,2024-02-19,2024-02-19
8,2024-02,20927,1556,"state, district, premise_type",shop code absent from frozen lookup,1,2024-02-05,2024-02-05
9,2024-02,20927,1558,"state, district, premise_type",shop code absent from frozen lookup,3,2024-02-05,2024-02-19


## Rebuild training and retain the original evaluation population

The original-history reconstruction must match the frozen 2025 training data before the comparison proceeds. The longer-history version uses the same calendar/coverage rules, with a January 2024 start. Original training rows and their labels must remain present; earlier history may change their rolling features and make additional rows eligible.

The 2024 cleaning audit reports invalid, missing-required-metadata, wholesale and frozen low-coverage-item removals separately. All source rows, including unmapped ones, remain in the selected-year cache. Missing descriptive-only fields (such as a shop display name) do not by themselves remove a localisable record. Item eligibility is intentionally not relearned here. Validation history remains originally prepared, holding evaluation inputs fixed for both versions.

In [5]:
def training_table(daily, start_month):
    """Same weekly median/coverage and exact-calendar history as Notebooks 04–05."""
    daily = daily.copy()
    daily["week_start"] = daily.date - pd.to_timedelta(daily.date.dt.weekday, unit="D")
    weekly = daily.groupby(keys).agg(weekly_median_price=("price", "median"),
        weekly_std_price=("price", "std"), record_count=("price", "size"),
        premise_count=("premise_code", "nunique"), observed_days=("date", "nunique")).reset_index()
    weekly = weekly.loc[(weekly.week_start >= pd.Timestamp(start_month + "-01"))
        & ((weekly.week_start + pd.Timedelta(days=6)) <= training_end)]
    parts = []
    for _, series in weekly.groupby(cfg.GROUP_KEYS, sort=False):
        grid = series.set_index("week_start").sort_index()
        grid = grid.reindex(pd.date_range(grid.index.min(), grid.index.max(), freq="W-MON"))
        price = grid.weekly_median_price
        for lag in [1, 2, 4]:
            grid[f"lag_{lag}w_price"] = price.shift(lag)
        for window in [4, 8]:
            grid[f"rolling_mean_{window}w"] = price.shift(1).rolling(window, min_periods=2).mean()
            grid[f"rolling_std_{window}w"] = price.shift(1).rolling(window, min_periods=2).std()
        grid["history_count_4w"] = price.shift(1).rolling(4, min_periods=1).count()
        grid["weekly_price_change_pct"] = (price / price.shift(1) - 1) * 100
        grid["change"] = (price.shift(-1) / price - 1) * 100
        current_ok = grid.record_count.ge(cfg.MIN_RECORDS) & grid.premise_count.ge(cfg.MIN_PREMISES)
        next_ok = grid.record_count.shift(-1).ge(cfg.MIN_RECORDS) & grid.premise_count.shift(-1).ge(cfg.MIN_PREMISES)
        eligible = current_ok & next_ok & grid.lag_1w_price.notna() & grid.history_count_4w.ge(cfg.MIN_HISTORY_4W)
        grid["surge_label"] = cfg.surge_mask(grid.change, settings["surge_threshold_pct"],
            settings["preparation"]["label_tolerance_pct"]).astype(int)
        part = grid.loc[price.notna() & eligible].copy()
        part.index.name = "week_start"
        parts.append(part.reset_index())
    result = pd.concat(parts, ignore_index=True)
    result["next_week_end"] = result.week_start + pd.Timedelta(days=13)
    result = result.loc[result.next_week_end.le(training_end)].copy()
    result["month"] = result.week_start.dt.month
    week_number = result.week_start.dt.isocalendar().week.astype(float)
    result["week_sin"] = np.sin(2 * np.pi * week_number / 52.1775)
    result["week_cos"] = np.cos(2 * np.pi * week_number / 52.1775)
    return result.sort_values(keys).reset_index(drop=True)

def model_inputs(frame):
    X = frame[features].copy()
    for column in settings["categorical_features"]:
        X[column] = X[column].astype("string").fillna("__missing__").astype(str)
    return X

def metrics(frame, scores):
    y = frame.surge_label.astype(int)
    predicted = scores >= selection["score_cutoff"]
    tn, fp, fn, tp = confusion_matrix(y, predicted, labels=[0, 1]).ravel()
    return {"rows": len(y), "positives": int(y.sum()),
        "average_precision": average_precision_score(y, scores) if y.nunique() == 2 else np.nan,
        "precision": precision_score(y, predicted, zero_division=0),
        "recall": recall_score(y, predicted, zero_division=0) if y.sum() else np.nan,
        "f1": f1_score(y, predicted, zero_division=0) if y.sum() else np.nan,
        "true_alerts": int(tp), "false_alerts": int(fp), "missed_surges": int(fn), "true_negatives": int(tn)}

## Fit the longer-history version and compare once

The original selected pipeline is cloned and preprocessing is refitted on the longer training period. Balanced weights retain their original policy; XGBoost's training-ratio weight, if applicable, is recomputed from the longer training labels. Both models use the original score cutoff. No search or test evaluation takes place.

A code/input fingerprint keeps each experiment separate. A completed matching cache reopens without fitting. Results are exploratory: gains are not permission to tune repeatedly against these same validation outcomes.

In [6]:
tracked = [model_dir / name for name in ["best_model.joblib", "model_manifest.json", "selection.json", "validation_metrics.csv"]]
tracked += [folder / name for name in ["experiment_settings.json", "train.parquet", "validation.parquet"]]
tracked += [cfg.STAGE3_MODEL_READY_PARQUET, cleaning_path, source_path, source_manifest_path, metadata_audit_path]
input_hashes = {str(p.resolve()): file_hash(p) for p in tracked}
notebook_path = cfg.notebook_path("12_longer_history_experiment.ipynb")
notebook_code = "\n".join("".join(c["source"]) for c in json.loads(notebook_path.read_text())["cells"] if c["cell_type"] == "code")
protocol = {"schema_version": 1, "status": "post-hoc exploratory longer-history validation comparison",
    "candidate": selection["candidate"], "cutoff": selection["score_cutoff"],
    "code_sha256": hashlib.sha256(notebook_code.encode()).hexdigest(), "input_sha256": input_hashes,
    "item_eligibility": "fixed original 2025 training choice; no new basket or exclusions",
    "historical_units_verified": False, "package_versions": {p: metadata.version(p) for p in packages}}
experiment_id = hashlib.sha256(json.dumps(protocol, sort_keys=True).encode()).hexdigest()[:16]
output_dir = cfg.PROJECT_ROOT / "12_longer_history_experiment" / experiment_id
output_dir.mkdir(parents=True, exist_ok=True)
cache_path = output_dir / "experiment_manifest.json"
artifact_names = ["longer_history_model.joblib", "training_comparison.csv", "training_group_support.csv",
    "historical_cleaning.csv", "validation_comparison.csv", "validation_group_comparison.csv", "validation_predictions.parquet"]
cache = json.loads(cache_path.read_text()) if cache_path.exists() else None
if cache is not None:
    if cache["protocol"] != protocol or set(cache["output_sha256"]) != set(artifact_names):
        raise ValueError("Longer-history cache does not match its declared protocol.")
    if any(not (output_dir / name).is_file() or file_hash(output_dir / name) != cache["output_sha256"][name] for name in artifact_names):
        raise ValueError("Derived longer-history artifacts changed. Restore their matching originals.")
    print("Reusing completed longer-history comparison; no downloads or fitting.")
else:
    original_train = pd.read_parquet(folder / "train.parquet").sort_values(keys).reset_index(drop=True)
    validation = pd.read_parquet(folder / "validation.parquet").sort_values(keys).reset_index(drop=True)
    if (validation.week_start + pd.Timedelta(days=6)).le(training_end).any() or validation.next_week_end.gt(validation_end).any():
        raise ValueError("Validation crosses the frozen chronological boundaries.")
    columns = cfg.GROUP_KEYS + ["date", "premise_code", "price"]
    daily_2025 = pd.read_parquet(cfg.STAGE3_MODEL_READY_PARQUET, columns=columns,
        filters=[("date", "<=", training_end)])
    daily_2025["date"] = pd.to_datetime(daily_2025.date)
    print("Checking original training reconstruction...")
    rebuilt_original = training_table(daily_2025, cfg.START_MONTH)
    comparison_columns = list(dict.fromkeys(keys + features + ["surge_label", "next_week_end"]))
    pd.testing.assert_frame_equal(original_train[comparison_columns], rebuilt_original[comparison_columns],
        check_dtype=False, check_exact=False, rtol=1e-9, atol=1e-9)
    earlier = pd.read_parquet(source_path, columns=columns)
    earlier["date"] = pd.to_datetime(earlier.date)
    if not earlier.date.between("2024-01-01", "2024-12-31").all():
        raise ValueError("Historical cache contains dates outside 2024.")
    invalid = earlier.price.isna() | ~np.isfinite(earlier.price) | earlier.price.le(0)
    missing_metadata = earlier[cfg.GROUP_KEYS].isna().any(axis=1) & ~invalid
    borong = (earlier.premise_type.eq("Borong").fillna(False) & ~invalid & ~missing_metadata
        if settings["preparation"]["remove_borong"] else pd.Series(False, index=earlier.index))
    low = earlier.item_code.isin(cleaning["excluded_item_codes"]) & ~invalid & ~missing_metadata & ~borong
    historical_cleaning = pd.DataFrame([{"source_selected_rows": len(earlier), "invalid_removed": int(invalid.sum()),
        "missing_required_metadata_removed": int(missing_metadata.sum()),
        "borong_removed": int(borong.sum()), "frozen_low_coverage_items_removed": int(low.sum()),
        "retained_daily_rows": int((~invalid & ~missing_metadata & ~borong & ~low).sum()),
        "statistical_flags_removed": 0, "historical_manual_exclusions": 0}])
    earlier = earlier.loc[~invalid & ~missing_metadata & ~borong & ~low].copy()
    combined = pd.concat([earlier, daily_2025], ignore_index=True)
    if combined.duplicated(["date", "premise_code", "item_code"]).any() or combined.date.gt(training_end).any():
        raise ValueError("Combined training source has duplicate keys or observations beyond training.")
    print("Building training features from 2024–2025...")
    longer_train = training_table(combined, "2024-01")
    if longer_train.surge_label.nunique() != 2:
        raise ValueError("Longer training needs both classes.")
    common = original_train.merge(longer_train, on=keys, how="inner", validate="one_to_one", suffixes=("_original", "_longer"))
    if len(common) != len(original_train) or not common.surge_label_original.eq(common.surge_label_longer).all():
        raise ValueError("Original training rows/labels were lost or changed; investigate integration before comparing.")
    training_rows, support_rows = [], []
    for version, frame in [("original_2025", original_train), ("longer_2024_2025", longer_train)]:
        training_rows.append({"version": version, "training_rows": len(frame), "surges": int(frame.surge_label.sum()),
            "surge_rate": float(frame.surge_label.mean()), "local_series": frame.groupby(cfg.GROUP_KEYS).ngroups,
            "rows_starting_2024": int(frame.week_start.dt.year.eq(2024).sum()),
            "rows_starting_2025": int(frame.week_start.dt.year.eq(2025).sum()),
            "first_week": str(frame.week_start.min().date()), "last_outcome": str(frame.next_week_end.max().date())})
        group_support = frame.groupby("project_group").surge_label.agg(rows="size", positives="sum").reset_index()
        group_support.insert(0, "version", version)
        support_rows.append(group_support)
    original_model = joblib.load(model_dir / "best_model.joblib")
    longer_model = clone(original_model)
    if selection["family"] == "XGBoost":
        longer_model.set_params(model__scale_pos_weight=(len(longer_train) - longer_train.surge_label.sum()) / longer_train.surge_label.sum())
    print("Fitting the longer-history model with the same settings...")
    longer_model.fit(model_inputs(longer_train), longer_train.surge_label.astype(int))
    scores = {"original_2025": original_model.predict_proba(model_inputs(validation))[:, 1],
        "longer_2024_2025": longer_model.predict_proba(model_inputs(validation))[:, 1]}
    comparison = pd.DataFrame([{"version": name, "cutoff": selection["score_cutoff"], **metrics(validation, values)} for name, values in scores.items()])
    reference = pd.read_csv(model_dir / "validation_metrics.csv")
    expected = reference.loc[reference.model.eq(selection["candidate"]), "average_precision"]
    if len(expected) != 1 or not np.isclose(comparison.iloc[0].average_precision, expected.iloc[0], rtol=1e-6, atol=1e-6):
        raise ValueError("Original validation AP does not reproduce; investigate before interpreting the comparison.")
    group_rows = []
    for group, part in validation.groupby("project_group"):
        for name, values in scores.items():
            group_rows.append({"project_group": group, "version": name, **metrics(part, values[part.index])})
    predictions = validation[keys + ["next_week_end", "surge_label"]].copy()
    for name, values in scores.items():
        predictions[name + "_score"] = values
    joblib.dump(longer_model, output_dir / "longer_history_model.joblib")
    pd.DataFrame(training_rows).to_csv(output_dir / "training_comparison.csv", index=False)
    pd.concat(support_rows, ignore_index=True).to_csv(output_dir / "training_group_support.csv", index=False)
    historical_cleaning.to_csv(output_dir / "historical_cleaning.csv", index=False)
    comparison.to_csv(output_dir / "validation_comparison.csv", index=False)
    pd.DataFrame(group_rows).to_csv(output_dir / "validation_group_comparison.csv", index=False)
    predictions.to_parquet(output_dir / "validation_predictions.parquet", index=False)
    if any(file_hash(Path(name)) != value for name, value in input_hashes.items()):
        raise ValueError("An upstream input changed during execution; results are incomplete.")
    cache = {"protocol": protocol, "prepared_test_accessed": False, "cutoff_retuned": False,
        "dashboard_model_replaced": False, "output_sha256": {name: file_hash(output_dir / name) for name in artifact_names}}
    temporary = cache_path.with_suffix(".tmp")
    temporary.write_text(json.dumps(cache, indent=2))
    temporary.replace(cache_path)
display(pd.read_csv(output_dir / "historical_cleaning.csv"))
display(pd.read_csv(output_dir / "training_comparison.csv"))
display(pd.read_csv(output_dir / "training_group_support.csv"))
comparison = pd.read_csv(output_dir / "validation_comparison.csv")
display(comparison)
display(pd.read_csv(output_dir / "validation_group_comparison.csv"))
print("Saved:", output_dir)
print("Exploratory longer-history comparison; original experiment, cutoff, test and dashboard model preserved.")

Checking original training reconstruction...
Building training features from 2024–2025...
Fitting the longer-history model with the same settings...


,source_selected_rows,invalid_removed,missing_required_metadata_removed,borong_removed,frozen_low_coverage_items_removed,retained_daily_rows,statistical_flags_removed,historical_manual_exclusions
0,2588232,0,23,1868,624,2585717,0,0


,version,training_rows,surges,surge_rate,local_series,rows_starting_2024,rows_starting_2025,first_week,last_outcome
0,original_2025,195664,20048,0.102461,6600,0,195664,2025-01-20,2025-12-28
1,longer_2024_2025,395757,38829,0.098113,6863,191166,204591,2024-01-15,2025-12-28


,version,project_group,rows,positives
0,original_2025,Chicken,19732,304
1,original_2025,Cooking Oil,12855,1
2,original_2025,Eggs,40195,1635
3,original_2025,Flour,7945,78
4,original_2025,Rice,4357,113
5,original_2025,Sugar,11976,0
6,original_2025,Vegetables,98604,17917
7,longer_2024_2025,Chicken,40135,581
8,longer_2024_2025,Cooking Oil,25591,17
9,longer_2024_2025,Eggs,80379,1712


,version,cutoff,rows,positives,average_precision,precision,recall,f1,true_alerts,false_alerts,missed_surges,true_negatives
0,original_2025,0.60065,43594,3912,0.223372,0.221820,0.657464,0.331721,2572,9023,1340,30659
1,longer_2024_2025,0.60065,43594,3912,0.329957,0.237831,0.690695,0.353827,2702,8659,1210,31023


,project_group,version,rows,positives,average_precision,precision,recall,f1,true_alerts,false_alerts,missed_surges,true_negatives
0,Chicken,original_2025,4884,25,0.132102,0.145833,0.280000,0.191781,7,41,18,4818
1,Chicken,longer_2024_2025,4884,25,0.187078,0.280000,0.280000,0.280000,7,18,18,4841
2,Cooking Oil,original_2025,2178,1,0.000548,0.000000,0.000000,0.000000,0,0,1,2177
3,Cooking Oil,longer_2024_2025,2178,1,0.000537,0.000000,0.000000,0.000000,0,0,1,2177
4,Eggs,original_2025,8892,53,0.076957,0.138462,0.169811,0.152542,9,56,44,8783
5,Eggs,longer_2024_2025,8892,53,0.063917,0.085366,0.132075,0.103704,7,75,46,8764
6,Flour,original_2025,1110,15,0.238534,0.250000,0.533333,0.340426,8,24,7,1071
7,Flour,longer_2024_2025,1110,15,0.227492,0.125000,0.066667,0.086957,1,7,14,1088
8,Rice,original_2025,384,2,0.514706,1.000000,0.500000,0.666667,1,0,1,382
9,Rice,longer_2024_2025,384,2,0.833333,0.500000,0.500000,0.500000,1,1,1,381


Saved: /Users/alfredyap/Documents/GitHub/FYP/data/runs/run_20261019_013241_618240/12_longer_history_experiment/3a31a8d36aad3a04
Exploratory longer-history comparison; original experiment, cutoff, test and dashboard model preserved.


## Interpret and stop before another adjustment

Compare AP, precision/recall/F1 at the fixed cutoff, false alerts, group support and historical cleaning attrition. More training rows or positives alone do not prove better forecasting. Additional history may introduce older regimes, source problems or seasonal information; this experiment cannot isolate the cause of a metric difference.

The fixed original basket excludes items judged sparse in 2025 even if they were better supported in 2024. Relearning that policy would be another experiment. Current lookup interpretation is provisional; whole-year integration checks do not verify the true price or historical units of every record. Current scores remain uncalibrated.

January–March 2026 and the original test have already been examined. Do not promote this model or make an independent performance claim from this comparison alone. Preserve the original experiment; freeze any proposed change before a future untouched evaluation period.

Share `training_comparison.csv`, `training_group_support.csv`, `historical_cleaning.csv`, `validation_comparison.csv` and `validation_group_comparison.csv`. No need to upload full source files. Reopening a completed matching run reuses cached data and results.

**Metadata exclusions:** `historical_metadata_review.csv` in the printed source-cache folder distinguishes absent shop codes from incomplete required fields and reports affected dates/items/counts. These 2024 records are retained as source evidence but cannot support a local-series model without metadata. Report this attrition; do not claim complete geographical coverage or infer the missing geography. The original study's frozen lookup and 2025 data are unchanged.
